# 🚴 Cyclistic Bike-Share Case Study

### Business Task

Analyze behavioral differences between casual riders and annual members using historical trip data to identify usage patterns associated with remaining a casual rider and support data-driven marketing decision-making aimed at increasing membership conversion.

### Importing and understanding general overview of data

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os


### Merging all 12 datasets

In [3]:
# Defining raw data files path 
data_folder = "../data/raw"

# Using glob to find all CSV files in that folder
all_files = glob.glob(os.path.join(data_folder, "*.csv"))

# Column validator
columns = ["ride_id", "rideable_type", "started_at", "ended_at", "start_station_name", "start_station_id",
            "end_station_name", "end_station_id", "start_lat", "start_lng", "end_lat", "end_lng", "member_casual"]

# Validating column names for files
validated_dfs = []
reference_dtypes = None
row_count = 0

for filename in all_files:
    df = pd.read_csv(filename)

    if list(df.columns) != columns:
        raise ValueError(f"Column mismatch in {filename}")
    
    # Set or Check Data Types
    if reference_dtypes is None:
        reference_dtypes = df.dtypes
        validated_dfs.append(df)
        row_count += len(df) 
    else:
        if df.dtypes.equals(reference_dtypes):
            validated_dfs.append(df)
            row_count += len(df)
        else:
            print(f"Mismatch in {filename}:")
            print(df.dtypes[df.dtypes != reference_dtypes])
            raise TypeError(f"Datatype mismatch in {filename}")

# Merging all validated files
combined_df = pd.concat(validated_dfs, ignore_index=True)

# Saving as merged_data.csv
output_filename = "../data/processed/merged_data.csv"

combined_df.to_csv(output_filename, index=False)

print(f"Successfully joined {len(validated_dfs)} files into '{output_filename}'!")

# Row count check
print(f"Indivdual file row_count: {row_count}")

Successfully joined 12 files into '../data/processed/merged_data.csv'!
Indivdual file row_count: 5552092


In [4]:
bike_data = pd.read_csv(output_filename)
print(f"Merged file row count: {len(bike_data)}")

Merged file row count: 5552092


In [5]:
bike_data.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,FD0EB1D32AF0D47E,classic_bike,2026-01-31 09:13:09.018,2026-01-31 09:28:10.302,Central St & Girard Ave,CHI02042,Dodge Ave & Church St,CHI00741,42.064313,-87.686152,42.048308,-87.698224,member
1,FB27405C3F8C824F,classic_bike,2026-01-15 14:25:42.526,2026-01-15 14:33:18.854,Shore Dr & 55th St,CHI00394,Woodlawn Ave & 55th St,CHI00423,41.795212,-87.580715,41.795264,-87.596471,casual
2,6FAFA1709403AA27,electric_bike,2026-01-06 12:55:33.572,2026-01-06 13:02:17.922,Hampden Ct & Diversey Pkwy,CHI02087,NaN,NaN,41.932470,-87.642420,41.940000,-87.640000,member
3,1F34C1FAD9FEC2D8,electric_bike,2026-01-26 16:22:25.011,2026-01-26 16:53:15.197,Carpenter St & Huron St,CHI00286,NaN,NaN,41.894532,-87.653412,41.830000,-87.670000,member
4,8E3E3072D8D3D918,electric_bike,2026-01-10 18:13:30.139,2026-01-10 19:31:56.971,Clinton St & Madison St,CHI00233,NaN,NaN,41.881660,-87.641150,41.890000,-87.630000,member


In [6]:
bike_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 5552092 entries, 0 to 5552091
Data columns (total 13 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ride_id             str    
 1   rideable_type       str    
 2   started_at          str    
 3   ended_at            str    
 4   start_station_name  str    
 5   start_station_id    str    
 6   end_station_name    str    
 7   end_station_id      str    
 8   start_lat           float64
 9   start_lng           float64
 10  end_lat             float64
 11  end_lng             float64
 12  member_casual       str    
dtypes: float64(4), str(9)
memory usage: 550.7 MB


In [7]:
bike_data.describe()

,start_lat,start_lng,end_lat,end_lng
count,5.552092e+06,5.552092e+06,5.546422e+06,5.546422e+06
mean,4.190372e+01,-8.764651e+01,4.190417e+01,-8.764685e+01
std,4.442616e-02,2.725586e-02,4.459733e-02,2.746200e-02
min,4.164850e+01,-8.789000e+01,4.149000e+01,-8.810000e+01
25%,4.188213e+01,-8.766000e+01,4.188241e+01,-8.766000e+01
50%,4.189859e+01,-8.764182e+01,4.189993e+01,-8.764275e+01
75%,4.193000e+01,-8.762998e+01,4.193000e+01,-8.763000e+01
max,4.207000e+01,-8.752000e+01,4.221000e+01,-8.742000e+01


### Data Cleaning

In [8]:
# Handling column names and Datatypes
bike_data.rename(columns={"started_at":"ride_start_datetime"}, inplace=True)
bike_data.rename(columns={"ended_at":"ride_end_datetime"}, inplace=True)

bike_data["ride_start_datetime"] = pd.to_datetime(bike_data["ride_start_datetime"])
bike_data["ride_end_datetime"] = pd.to_datetime(bike_data["ride_end_datetime"])

In [9]:
bike_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 5552092 entries, 0 to 5552091
Data columns (total 13 columns):
 #   Column               Dtype         
---  ------               -----         
 0   ride_id              str           
 1   rideable_type        str           
 2   ride_start_datetime  datetime64[us]
 3   ride_end_datetime    datetime64[us]
 4   start_station_name   str           
 5   start_station_id     str           
 6   end_station_name     str           
 7   end_station_id       str           
 8   start_lat            float64       
 9   start_lng            float64       
 10  end_lat              float64       
 11  end_lng              float64       
 12  member_casual        str           
dtypes: datetime64[us](2), float64(4), str(7)
memory usage: 550.7 MB


In [10]:
# Dropping duplicate rows if present
bike_data.drop_duplicates(inplace=True)
bike_data.shape

(5552092, 13)

In [11]:
# Handling missing values
bike_data.isnull().sum()

ride_id                      0
rideable_type                0
ride_start_datetime          0
ride_end_datetime            0
start_station_name     1186374
start_station_id       1186374
end_station_name       1246827
end_station_id         1246827
start_lat                    0
start_lng                    0
end_lat                   5670
end_lng                   5670
member_casual                0
dtype: int64

In [12]:
# 1. Rounding coordinates to 4 decimal places (~11m precision) 
# This helps group bikes parked slightly differently at the same station.
bike_data['start_lat_round'] = bike_data['start_lat'].round(4)
bike_data['start_lng_round'] = bike_data['start_lng'].round(4)
bike_data['end_lat_round'] = bike_data['end_lat'].round(4)
bike_data['end_lng_round'] = bike_data['end_lng'].round(4)

# 2. Building the Master Lookup Table
# Looking at all known station/coordinate pairs to build the "map"
start_lookup = bike_data.dropna(subset=['start_station_name'])[['start_lat_round', 'start_lng_round', 'start_station_name']]
end_lookup = bike_data.dropna(subset=['end_station_name'])[['end_lat_round', 'end_lng_round', 'end_station_name']]

# Renaming columns so they can be merged into one master reference
start_lookup.columns = ['lat', 'lng', 'station_name']
end_lookup.columns = ['lat', 'lng', 'station_name']

# Combining both and keeping the most frequent name for each coordinate pair
master_lookup = pd.concat([start_lookup, end_lookup]).drop_duplicates(subset=['lat', 'lng'])

# 3. Recovering Start Station Names
bike_data = bike_data.merge(master_lookup, left_on=['start_lat_round', 'start_lng_round'], 
                            right_on=['lat', 'lng'], how='left', suffixes=('', '_recovered'))

bike_data['start_station_name'] = bike_data['start_station_name'].fillna(bike_data['station_name'])
bike_data.drop(columns=['lat', 'lng', 'station_name'], inplace=True)

# 4. Recovering End Station Names
bike_data = bike_data.merge(master_lookup, left_on=['end_lat_round', 'end_lng_round'], 
                            right_on=['lat', 'lng'], how='left', suffixes=('', '_recovered'))

bike_data['end_station_name'] = bike_data['end_station_name'].fillna(bike_data['station_name'])
bike_data.drop(columns=['lat', 'lng', 'station_name', 'start_lat_round', 'start_lng_round', 'end_lat_round', 'end_lng_round'], inplace=True)

# 5. Final Clean up
# Filling remaining NaNs with a placeholder
bike_data['start_station_name'] = bike_data['start_station_name'].fillna("Public Rack/Unknown")
bike_data['end_station_name'] = bike_data['end_station_name'].fillna("Public Rack/Unknown")

# Dropping rows where end coordinates are missing (cannot deduce anything)
bike_data.dropna(subset=['end_lat', 'end_lng'], inplace=True)

print(f"Final dataset size: {bike_data.shape}")
print(bike_data.isnull().sum())

Final dataset size: (5546422, 13)
ride_id                      0
rideable_type                0
ride_start_datetime          0
ride_end_datetime            0
start_station_name           0
start_station_id       1186374
end_station_name             0
end_station_id         1241157
start_lat                    0
start_lng                    0
end_lat                      0
end_lng                      0
member_casual                0
dtype: int64


In [13]:
# 1. Dropping the redundant ID columns
bike_data.drop(columns=['start_station_id', 'end_station_id'], inplace=True)

# 2. Calculating Ride Length
bike_data['ride_length_mins'] = (bike_data['ride_end_datetime'] - bike_data['ride_start_datetime']).dt.total_seconds() / 60

# Creating derived time featues
bike_data["day_of_week"] = bike_data["ride_start_datetime"].dt.day_of_week
bike_data["month"] = bike_data["ride_start_datetime"].dt.month_name()
bike_data["hour"] = bike_data["ride_start_datetime"].dt.hour

# 3. Filtering out the "Noise"
# Trips under 1 minute are usually accidental starts/locks
# Trips over 24 hours are usually stolen or lost bikes
original_count = len(bike_data)
bike_data = bike_data[(bike_data['ride_length_mins'] > 1) & (bike_data['ride_length_mins'] < 1440)]

print(f"Removed {original_count - len(bike_data)} outlier rows.")
print(f"Final cleaned row count: {len(bike_data)}")

Removed 148610 outlier rows.
Final cleaned row count: 5397812


In [14]:
output_path = "../data/processed/cleaned_bike_data.csv"

# index=False prevents pandas from adding an extra 'Unnamed: 0' column
bike_data.to_csv(output_path, index=False)

print(f"Cleaned data saved to {output_path}")

Cleaned data saved to ../data/processed/cleaned_bike_data.csv


In [52]:
cleaned_data = pd.read_csv("../data/processed/cleaned_bike_data.csv")
cleaned_data.head()

,ride_id,rideable_type,ride_start_datetime,ride_end_datetime,start_station_name,end_station_name,start_lat,start_lng,end_lat,end_lng,member_casual,ride_length_mins,day_of_week,month,hour
0,FD0EB1D32AF0D47E,classic_bike,2026-01-31 09:13:09.018,2026-01-31 09:28:10.302,Central St & Girard Ave,Dodge Ave & Church St,42.064313,-87.686152,42.048308,-87.698224,member,15.021400,5,January,9
1,FB27405C3F8C824F,classic_bike,2026-01-15 14:25:42.526,2026-01-15 14:33:18.854,Shore Dr & 55th St,Woodlawn Ave & 55th St,41.795212,-87.580715,41.795264,-87.596471,casual,7.605467,3,January,14
2,6FAFA1709403AA27,electric_bike,2026-01-06 12:55:33.572,2026-01-06 13:02:17.922,Hampden Ct & Diversey Pkwy,Public Rack/Unknown,41.932470,-87.642420,41.940000,-87.640000,member,6.739167,1,January,12
3,1F34C1FAD9FEC2D8,electric_bike,2026-01-26 16:22:25.011,2026-01-26 16:53:15.197,Carpenter St & Huron St,Public Rack/Unknown,41.894532,-87.653412,41.830000,-87.670000,member,30.836433,0,January,16
4,8E3E3072D8D3D918,electric_bike,2026-01-10 18:13:30.139,2026-01-10 19:31:56.971,Clinton St & Madison St,Public Rack/Unknown,41.881660,-87.641150,41.890000,-87.630000,member,78.447200,5,January,18


In [53]:
cleaned_data.head()

,ride_id,rideable_type,ride_start_datetime,ride_end_datetime,start_station_name,end_station_name,start_lat,start_lng,end_lat,end_lng,member_casual,ride_length_mins,day_of_week,month,hour
0,FD0EB1D32AF0D47E,classic_bike,2026-01-31 09:13:09.018,2026-01-31 09:28:10.302,Central St & Girard Ave,Dodge Ave & Church St,42.064313,-87.686152,42.048308,-87.698224,member,15.021400,5,January,9
1,FB27405C3F8C824F,classic_bike,2026-01-15 14:25:42.526,2026-01-15 14:33:18.854,Shore Dr & 55th St,Woodlawn Ave & 55th St,41.795212,-87.580715,41.795264,-87.596471,casual,7.605467,3,January,14
2,6FAFA1709403AA27,electric_bike,2026-01-06 12:55:33.572,2026-01-06 13:02:17.922,Hampden Ct & Diversey Pkwy,Public Rack/Unknown,41.932470,-87.642420,41.940000,-87.640000,member,6.739167,1,January,12
3,1F34C1FAD9FEC2D8,electric_bike,2026-01-26 16:22:25.011,2026-01-26 16:53:15.197,Carpenter St & Huron St,Public Rack/Unknown,41.894532,-87.653412,41.830000,-87.670000,member,30.836433,0,January,16
4,8E3E3072D8D3D918,electric_bike,2026-01-10 18:13:30.139,2026-01-10 19:31:56.971,Clinton St & Madison St,Public Rack/Unknown,41.881660,-87.641150,41.890000,-87.630000,member,78.447200,5,January,18


### Analysis

Ride Length

In [54]:
# Riding length patterns between members
cleaned_data.groupby("member_casual")["ride_length_mins"].describe()

,count,mean,std,min,25%,50%,75%,max
member_casual,,,,,,,,
casual,1916136.0,19.877889,39.213845,1.000017,6.820983,11.892817,21.676358,1439.975950
member,3481676.0,12.228021,21.214955,1.000017,5.219133,8.751167,14.698737,1439.827217


In [55]:
# Overall percentage of casual rides
casual_rides = (cleaned_data["member_casual"] == "casual").sum()
percent = (casual_rides / len(bike_data)) * 100 
print(f"{round(percent, 2)}% of the rides are casual")

35.5% of the rides are casual


In [56]:
# Overall percentage of rides over 30 mins that are casual
rides_over_30 = (cleaned_data["ride_length_mins"] > 30).sum()
rides_over_30_casual = ((cleaned_data["member_casual"] == "casual") & (cleaned_data["ride_length_mins"] > 30)).sum()
percent = (rides_over_30_casual / rides_over_30) * 100
print(f"{round(percent, 2)}% of the rides over 30 minutes are casual")

61.18% of the rides over 30 minutes are casual


Day of week

In [57]:
# Mapping day_of_week numbers to respective days

# Ensuring correct data type
cleaned_data["day_of_week"] = cleaned_data["day_of_week"].astype("int64")

day_map = {
    0: "Monday", 
    1: "Tuesday", 
    2: "Wednesday", 
    3: "Thursday", 
    4: "Friday", 
    5: "Saturday", 
    6: "Sunday"
}

# Replace the numbers with words
cleaned_data["day_of_week"] = cleaned_data["day_of_week"].map(day_map)

# Quick check to see the change
cleaned_data["day_of_week"].value_counts()

day_of_week
Saturday     833787
Friday       823178
Thursday     807354
Tuesday      776726
Wednesday    750301
Monday       714549
Sunday       691917
Name: count, dtype: int64

In [58]:
cleaned_data.head()

,ride_id,rideable_type,ride_start_datetime,ride_end_datetime,start_station_name,end_station_name,start_lat,start_lng,end_lat,end_lng,member_casual,ride_length_mins,day_of_week,month,hour
0,FD0EB1D32AF0D47E,classic_bike,2026-01-31 09:13:09.018,2026-01-31 09:28:10.302,Central St & Girard Ave,Dodge Ave & Church St,42.064313,-87.686152,42.048308,-87.698224,member,15.021400,Saturday,January,9
1,FB27405C3F8C824F,classic_bike,2026-01-15 14:25:42.526,2026-01-15 14:33:18.854,Shore Dr & 55th St,Woodlawn Ave & 55th St,41.795212,-87.580715,41.795264,-87.596471,casual,7.605467,Thursday,January,14
2,6FAFA1709403AA27,electric_bike,2026-01-06 12:55:33.572,2026-01-06 13:02:17.922,Hampden Ct & Diversey Pkwy,Public Rack/Unknown,41.932470,-87.642420,41.940000,-87.640000,member,6.739167,Tuesday,January,12
3,1F34C1FAD9FEC2D8,electric_bike,2026-01-26 16:22:25.011,2026-01-26 16:53:15.197,Carpenter St & Huron St,Public Rack/Unknown,41.894532,-87.653412,41.830000,-87.670000,member,30.836433,Monday,January,16
4,8E3E3072D8D3D918,electric_bike,2026-01-10 18:13:30.139,2026-01-10 19:31:56.971,Clinton St & Madison St,Public Rack/Unknown,41.881660,-87.641150,41.890000,-87.630000,member,78.447200,Saturday,January,18


In [60]:
# 1. Calculate percentage distribution for each group separately
casual_dist = cleaned_data[cleaned_data['member_casual'] == 'casual']['day_of_week'].value_counts(normalize=True) * 100
member_dist = cleaned_data[cleaned_data['member_casual'] == 'member']['day_of_week'].value_counts(normalize=True) * 100

# 2. Combine them into a single DataFrame
final_comparison = pd.DataFrame({
    'casual_pct': casual_dist,
    'member_pct': member_dist
})

# 3. Sort by day order
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
final_comparison = final_comparison.reindex(day_order)

print("--- Weekday Usage Distribution (%) ---")
print(final_comparison.round(2))

--- Weekday Usage Distribution (%) ---
             casual_pct  member_pct
day_of_week                        
Monday            11.47       14.21
Tuesday           11.40       16.04
Wednesday         11.05       15.47
Thursday          12.90       16.09
Friday            15.99       14.84
Saturday          20.63       12.59
Sunday            16.57       10.75
